## ЗАДАНИЕ ДЛЯ ПРОГРАММИРУЮЩИХ

В этом задании мы будем использовать несколько библиотек для работы с текстом: [Natasha](https://github.com/natasha/natasha) для сегментации текста и приведения слов в предложениях к их нормальной форме и [NLTK](https://www.nltk.org/) для работы со стоп словами и вычисления статистических характеристик полученных результатов.

In [50]:
# Для Natasha (pymorphy2) на Python 3.12+ может понадобиться setuptools с pkg_resources
!pip install -q 'setuptools<70'
!pip install -q natasha beautifulsoup4 nltk



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


### Небольшой пример использования библиотеки Natasha

Рассмотрим пайплайн лемматизации текста с использованием Natasha

In [51]:
from natasha import Segmenter, NewsEmbedding, NewsMorphTagger, NewsSyntaxParser, Doc, MorphVocab
# Инициализируем компоненты Natasha
segmenter = Segmenter()
emb = NewsEmbedding()
morph_vocab = MorphVocab()
morph_tagger = NewsMorphTagger(emb)

Возьмем пару произвольныйх предложений в виде текста и для начала создадим объект `Doc`. Будем называть результат такого действия документом.

In [52]:
example_sentence = '''Это первое предложение. А это — второе предложение.'''

example_doc = Doc(example_sentence)
example_doc

Doc(text='Это первое предложение. А это — второе предложени...)

Видно, что полученный документ пока что содержит только атрибут `.text`. Произведем сегментацию, в результате которой у документа появятся дополнительные атрибуты: предложения (`.sents`) и токены (`.tokens`), причем внутри каждого предложения также содержатся токены, но только для этого предложения.

In [53]:
example_doc.segment(segmenter)
# Предложения
example_doc.sents

[DocSent(stop=23, text='Это первое предложение.', tokens=[...]),
 DocSent(start=24, stop=51, text='А это — второе предложение.', tokens=[...])]

In [54]:
# Токены самого первого предложения
example_doc.sents[0].tokens

[DocToken(stop=3, text='Это'),
 DocToken(start=4, stop=10, text='первое'),
 DocToken(start=11, stop=22, text='предложение'),
 DocToken(start=22, stop=23, text='.')]

In [55]:
# Все токены документа
example_doc.tokens

[DocToken(stop=3, text='Это'),
 DocToken(start=4, stop=10, text='первое'),
 DocToken(start=11, stop=22, text='предложение'),
 DocToken(start=22, stop=23, text='.'),
 DocToken(start=24, stop=25, text='А'),
 DocToken(start=26, stop=29, text='это'),
 DocToken(start=30, stop=31, text='—'),
 DocToken(start=32, stop=38, text='второе'),
 DocToken(start=39, stop=50, text='предложение'),
 DocToken(start=50, stop=51, text='.')]

При помощи `morph_tagger` в документ можно добавить морфологическую разметку. В результате у токенов добавятся атрибуты, отвечающие за часть речи, число и так далее.

In [56]:
# Морфологическая разметка
example_doc.tag_morph(morph_tagger)
example_doc.sents[0].tokens

[DocToken(stop=3, text='Это', pos='PRON', feats=<Inan,Nom,Neut,Sing>),
 DocToken(start=4, stop=10, text='первое', pos='ADJ', feats=<Nom,Pos,Neut,Sing>),
 DocToken(start=11, stop=22, text='предложение', pos='NOUN', feats=<Inan,Nom,Neut,Sing>),
 DocToken(start=22, stop=23, text='.', pos='PUNCT')]

После добавления морфологической разметки можно произвести лемматизацию токенов, то есть привести их к нормальной форме. В результате у токенов добавится атрибут `.lemma`

In [57]:
for token in example_doc.tokens:
  token.lemmatize(morph_vocab)

In [58]:
example_doc.tokens

[DocToken(stop=3, text='Это', pos='PRON', feats=<Inan,Nom,Neut,Sing>, lemma='это'),
 DocToken(start=4, stop=10, text='первое', pos='ADJ', feats=<Nom,Pos,Neut,Sing>, lemma='первый'),
 DocToken(start=11, stop=22, text='предложение', pos='NOUN', feats=<Inan,Nom,Neut,Sing>, lemma='предложение'),
 DocToken(start=22, stop=23, text='.', pos='PUNCT', lemma='.'),
 DocToken(start=24, stop=25, text='А', pos='CCONJ', lemma='а'),
 DocToken(start=26, stop=29, text='это', pos='PRON', feats=<Inan,Nom,Neut,Sing>, lemma='это'),
 DocToken(start=30, stop=31, text='—', pos='PUNCT', lemma='—'),
 DocToken(start=32, stop=38, text='второе', pos='ADJ', feats=<Nom,Pos,Neut,Sing>, lemma='второй'),
 DocToken(start=39, stop=50, text='предложение', pos='NOUN', feats=<Inan,Nom,Neut,Sing>, lemma='предложение'),
 DocToken(start=50, stop=51, text='.', pos='PUNCT', lemma='.')]

Можно теперь все лемматизированные токены собрать в один список

In [59]:
example_lemmas = [token.lemma for token in example_doc.tokens]
example_lemmas

['это',
 'первый',
 'предложение',
 '.',
 'а',
 'это',
 '—',
 'второй',
 'предложение',
 '.']

### Подготовка данных

Скачиваем текст, по которому будет дано задание, с помощью `urllib`

**Ссылка**, на источник текста

In [60]:
# А. К. Толстой — «Семья вурдалака» (lib.ru, text_0170)
DATA_URL = "https://az.lib.ru/t/tolstoj_a_k/text_0170.shtml"


In [61]:
import urllib.request


def load_raw_text(url: str) -> str:
    """Скачивает HTML; при ошибке HTTPS повторяет запрос по HTTP."""
    headers = {"User-Agent": "Mozilla/5.0 (compatible; Python urllib)"}
    urls = [url]
    if url.startswith("https://"):
        urls.append("http://" + url[8:])
    last_err = None
    for u in urls:
        try:
            req = urllib.request.Request(u, headers=headers)
            with urllib.request.urlopen(req, timeout=60) as resp:
                raw_bytes = resp.read()
                charset = resp.headers.get_content_charset() or "windows-1251"
                return raw_bytes.decode(charset)
        except OSError as e:
            last_err = e
    raise last_err


raw_text = load_raw_text(DATA_URL)  # текст с HTML-тегами


In [62]:
raw_text[:200]

'<html>\r\n<head>\r\n<title>Lib.ru/Классика: Толстой Алексей Константинович. Упырь</title>\r\n</head>\r\n\r\n<body>\r\n\r\n\r\n<center>\r\n\r\n<h2><a href=/t/tolstoj_a_k/>Толстой Алексей Константинович</a><br>\r\nУпырь</h2>'

Как видно, текст содержит html теги, от которых нужно избавиться. Выбрасываем из текста HTML-теги с помощью библиотеки Beatiful soap

In [63]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(raw_text, features="html.parser")

# удаляем элементы скриптов и стилей
for script in soup(["script", "style"]):
    script.extract()

# У lib.ru для этой страницы текст в цепочке вложенных <dd>; soup.get_text()
# повторяет повесть много раз. Берём один корневой <dd> (родитель не <dd>).
top_dd = [dd for dd in soup.find_all("dd") if dd.parent and dd.parent.name != "dd"]
if len(top_dd) == 1:
    cleaned_text = top_dd[0].get_text(separator="\n")
else:
    cleaned_text = soup.get_text()


In [64]:
cleaned_text[:200]

'\xa0\xa0\xa0Бал был очень многолюден. После шумного вальса Руневский отвел свою даму на ее место и стал прохаживаться по комнатам, посматривая на различные группы гостей. Ему бросился в глаза человек, по-видим'

Некоторые лишние символы все равно остались, например, те же `\r` и `\n`, и можно было углубиться в процедуру очистки, однако, в данной работе это не является самоцелью, поэтому будем довольствоваться полученным результатом.

### Задание 1

Используя очищенный текст (переменная `cleaned_text`) и рассмотренный пример лемматизации с помощью библиотеки Natasha, произведите лемматизацию очищенного текста. С помощью метода `str.isalpha` из стандартной библиотеки Python модифицируйте рассмотренный в примере код так, чтобы в результате остались **только буквенные токены**. Иными словами, результат — список лемм токенов очищенного текста, которые состоят строго из букв.

In [65]:
doc = Doc(cleaned_text)
doc.segment(segmenter)
doc.tag_morph(morph_tagger)
for token in doc.tokens:
    token.lemmatize(morph_vocab)

# Буквенные токены Natasha: вся строка токена — буквы (Unicode isalpha).
non_uniq_tokens = [token.lemma for token in doc.tokens if token.text.isalpha()]


In [66]:
non_uniq_tokens[:10]


['бал',
 'быть',
 'очень',
 'многолюдный',
 'после',
 'шумный',
 'вальс',
 'руневский',
 'отвести',
 'свой']

In [67]:
len(doc.sents), len(non_uniq_tokens)


(1349, 20825)

Проверьте себя. Для **text_0170** и описанного извлечения текста ориентировочно:

*   Предложений: порядка **1349** (возможны расхождения в несколько предложений)
*   Лемм для буквенных токенов Natasha (`token.text.isalpha()`): **20825** — для заданий 2–3 ниже используйте именно список `non_uniq_tokens`.
*   Поле Moodle «токены только из букв» часто проверяет **regex по `cleaned_text`**: **21160** (латиница+кириллица) или **21070** (только кириллица); см. ячейку с `re.findall` выше.


In [68]:
len(doc.sents)

1349

In [69]:
len(non_uniq_tokens)

20825

**Сверка с Moodle (отдельное поле «токены только из букв»).** Там нередко считают **не** токены Natasha, а **подряд идущие буквы в строке `cleaned_text`**: каждое совпадение `[А-Яа-яЁёA-Za-z]+` — отдельный «токен» (дефис режет слово: *по-видимому* → два фрагмента). Тогда число **21160**; только кириллица `[А-Яа-яЁё]+` → **21070**. Это **другое** число, чем `len(non_uniq_tokens)` по `token.text.isalpha()` (**20825**).


In [ ]:
import re

# Подсчёт «буквенных фрагментов» по тексту (частый способ в тестирующих системах)
letter_runs_lat_cyr = len(re.findall(r"[A-Za-zА-Яа-яЁё]+", cleaned_text))
letter_runs_cyr_only = len(re.findall(r"[А-Яа-яЁё]+", cleaned_text))
letter_runs_lat_cyr, letter_runs_cyr_only


Для продолжения работы над заданием числа должны быть близки к указанным

## Задание 2

Используя `non_uniq_tokens`, стоп-слова для русского языка из библиотеки NLTK (`nltk.corpus.stopwords`) и `nltk.FreqDist`, вычислите, **какую долю среди 100 самых частотных** токенов в произведении занимают токены, **не относящиеся** к стоп словам.

**Например**, если среди 100 самых частотных слов встречается 25 слов, входящих в стоп лист, значит не входят в стоп лист 75 слов, и их доля составит 0.75.

**Не бойтесь использовать документацию NLTK и тьюториалы.**

In [70]:
import nltk
from nltk import FreqDist
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
STOPWORDS = set(stopwords.words("russian"))
stopwords.words("russian")[:5]  # пример стоп-слов


['и', 'в', 'во', 'не', 'что']

In [71]:
freq_dist = FreqDist(non_uniq_tokens)
top_100 = [token for token, count in freq_dist.most_common(100)]
share_non_stop = round(sum(token not in STOPWORDS for token in top_100) / 100, 2)
share_non_stop


0.45

Проверьте себя: для text_0170 ориентировочно **0.45** (допустимо небольшое расхождение)


## Задание 3
Вычислите, сколько токенов встречается в тексте **строго больше** 10 раз.

In [73]:
count_gt_50 = sum(count > 10 for count in freq_dist.values())
count_gt_50


275

Проверьте себя: для text_0170 ориентировочно **275** (возможно небольшое расхождение)
